## Part A - Scikit-learn Implementation 


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
    r2_score,accuracy_score,precision_score,recall_score,f1_score)

C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [5]:
df = pd.read_csv("garments_worker_productivity.csv")
print(df.head())
print("--------------")
print(df.shape)
print("--------------")
print(df.columns)

       date   quarter  department       day  team  targeted_productivity  \
0  1/1/2015  Quarter1      sweing  Thursday     8                   0.80   
1  1/1/2015  Quarter1  finishing   Thursday     1                   0.75   
2  1/1/2015  Quarter1      sweing  Thursday    11                   0.80   
3  1/1/2015  Quarter1      sweing  Thursday    12                   0.80   
4  1/1/2015  Quarter1      sweing  Thursday     6                   0.80   

     smv     wip  over_time  incentive  idle_time  idle_men  \
0  26.16  1108.0       7080         98        0.0         0   
1   3.94     NaN        960          0        0.0         0   
2  11.41   968.0       3660         50        0.0         0   
3  11.41   968.0       3660         50        0.0         0   
4  25.90  1170.0       1920         50        0.0         0   

   no_of_style_change  no_of_workers  actual_productivity  
0                   0           59.0             0.940725  
1                   0            8.0        

In [6]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df["meets_target"] = (df["actual_productivity"] >= df["targeted_productivity"]).astype(int)
print(df[["actual_productivity", "targeted_productivity", "meets_target"]].head())

   actual_productivity  targeted_productivity  meets_target
0             0.940725                   0.80             1
1             0.886500                   0.75             1
2             0.800570                   0.80             1
3             0.800570                   0.80             1
4             0.800382                   0.80             1


In [7]:
X = df.drop(columns=["actual_productivity", "meets_target"])
y_reg = df["actual_productivity"]
y_class = df["meets_target"]

print("Features:")
print(X.columns)

print("\nRegression target:")
print(y_reg.name)

print("\nClassification target:")
print(y_class.name)

Features:
Index(['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity',
       'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men',
       'no_of_style_change', 'no_of_workers'],
      dtype='str')

Regression target:
actual_productivity

Classification target:
meets_target


In [8]:
if "date" in X.columns:
    X = X.drop(columns=["date"])
    print(X.columns)

Index(['quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv',
       'wip', 'over_time', 'incentive', 'idle_time', 'idle_men',
       'no_of_style_change', 'no_of_workers'],
      dtype='str')


In [14]:
numeric_columns = [
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

categorical_columns = [
    "quarter",
    "department",
    "day",
    "team"
]

print("Numerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical columns:
['quarter', 'department', 'day', 'team']


In [15]:
train_indices, test_indices = train_test_split(np.arange(len(X)),
    test_size=0.20,
    random_state=42)

X_train = X.iloc[train_indices]
X_test = X.iloc[test_indices]

y_reg_train = y_reg.iloc[train_indices]
y_reg_test = y_reg.iloc[test_indices]

y_class_train = y_class.iloc[train_indices]
y_class_test = y_class.iloc[test_indices]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 957
Testing samples: 240


In [16]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

In [17]:
regression_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

In [18]:
start_time = time.perf_counter()
regression_model.fit(X_train, y_reg_train)
regression_training_time = time.perf_counter() - start_time

print("Linear Regression training time:", regression_training_time, "seconds")

Linear Regression training time: 0.12997949999999037 seconds


In [19]:
start_time = time.perf_counter()
y_reg_pred = regression_model.predict(X_test)
regression_prediction_time = time.perf_counter() - start_time
mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)
print("Linear Regression Results")
print("----------")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("Training time:", regression_training_time, "seconds")
print("Prediction time:", regression_prediction_time, "seconds")

Linear Regression Results
----------
MAE: 0.11211070233876953
RMSE: 0.15050400932972155
R2: 0.14691703672158807
Training time: 0.12997949999999037 seconds
Prediction time: 0.015139099999942118 seconds


In [20]:
classification_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))])

In [21]:
start_time = time.perf_counter()
classification_model.fit(X_train, y_class_train)
classification_training_time = time.perf_counter() - start_time
print("Logistic Regression training time:",classification_training_time,"seconds")

Logistic Regression training time: 0.06973910000033356 seconds


In [22]:
start_time = time.perf_counter()
y_class_pred = classification_model.predict(X_test)
classification_prediction_time = time.perf_counter() - start_time
accuracy = accuracy_score(y_class_test, y_class_pred)
precision = precision_score(y_class_test, y_class_pred)
recall = recall_score(y_class_test, y_class_pred)
f1 = f1_score(y_class_test, y_class_pred)
print("Logistic Regression Results")
print("---------------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Training time:", classification_training_time, "seconds")
print("Prediction time:", classification_prediction_time, "seconds")

Logistic Regression Results
---------------------------
Accuracy: 0.7625
Precision: 0.7912621359223301
Recall: 0.9209039548022598
F1-score: 0.8511749347258486
Training time: 0.06973910000033356 seconds
Prediction time: 0.017576400000052672 seconds


In [23]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Logistic Regression"],
    "Training Time (seconds)": [
        regression_training_time,
        classification_training_time],
    "Prediction Time (seconds)": [
        regression_prediction_time,
        classification_prediction_time],
    "MAE": [mae,np.nan],
    "RMSE": [rmse,np.nan],
    "R2": [r2,np.nan],
    "Accuracy": [np.nan,accuracy],
    "Precision": [np.nan, precision],
    "Recall": [np.nan,recall],
    "F1-score": [ np.nan,f1]})

print(results)

                 Model  Training Time (seconds)  Prediction Time (seconds)  \
0    Linear Regression                 0.129979                   0.015139   
1  Logistic Regression                 0.069739                   0.017576   

        MAE      RMSE        R2  Accuracy  Precision    Recall  F1-score  
0  0.112111  0.150504  0.146917       NaN        NaN       NaN       NaN  
1       NaN       NaN       NaN    0.7625   0.791262  0.920904  0.851175  


## Part B - From-Scratch Implementation 


In [24]:
# For removing target columns
X = df.drop(columns=["actual_productivity", "meets_target"])

if "date" in X.columns:
    X = X.drop(columns=["date"])

y_reg = df["actual_productivity"].values
y_class = df["meets_target"].values

# Reuse the SAME train-test samples from Part A
X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_reg_train = y_reg[train_indices]
y_reg_test = y_reg[test_indices]

y_class_train = y_class[train_indices]
y_class_test = y_class[test_indices]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 957
Testing samples: 240


In [25]:
numeric_columns = [
    "targeted_productivity", "smv", "wip", "over_time",
    "incentive", "idle_time", "idle_men",
    "no_of_style_change", "no_of_workers"
]

categorical_columns = [
    "quarter", "department", "day", "team"
]

# Missing numerical values then training median
for col in numeric_columns:
    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

# Missing categorical values then training mode
for col in categorical_columns:
    mode_value = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_value)
    X_test[col] = X_test[col].fillna(mode_value)

# Onehot encoding
X_train = pd.get_dummies(X_train, columns=categorical_columns, dtype=float)
X_test = pd.get_dummies(X_test, columns=categorical_columns, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print("Features after encoding:", X_train.shape[1])
print(X_train.head())

Features after encoding: 35
      targeted_productivity    smv     wip  over_time  incentive  idle_time  \
1189                   0.70  30.48   914.0       6840         30        0.0   
575                    0.75   3.94  1023.0       2280          0        0.0   
76                     0.75   2.90  1023.0        960          0        0.0   
731                    0.70   4.15  1023.0       1800          0        0.0   
138                    0.80  11.61   548.0      15120         63        0.0   

      idle_men  no_of_style_change  no_of_workers  quarter_Quarter1  ...  \
1189         0                   1           57.0               0.0  ...   
575          0                   0           19.0               1.0  ...   
76           0                   0            8.0               1.0  ...   
731          0                   0           15.0               0.0  ...   
138          0                   0           31.5               0.0  ...   

      team_3  team_4  team_5  team_6  te

In [26]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
train_std[train_std == 0] = 1

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

X_train_scaled = X_train_scaled.values
X_test_scaled = X_test_scaled.values

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Training shape: (957, 35)
Testing shape: (240, 35)


In [27]:
import numpy as np
import time

X_train_lr = np.column_stack([np.ones(X_train_scaled.shape[0]),X_train_scaled])
X_test_lr = np.column_stack([np.ones(X_test_scaled.shape[0]),X_test_scaled])

start_time = time.perf_counter()
beta = np.linalg.pinv(
    X_train_lr.T @ X_train_lr
) @ X_train_lr.T @ y_reg_train

regression_training_time = time.perf_counter() - start_time
print("Linear Regression training time:", regression_training_time, "seconds")

Linear Regression training time: 0.2233992000001308 seconds


In [28]:
start_time = time.perf_counter()
y_reg_pred = X_test_lr @ beta
regression_prediction_time = time.perf_counter() - start_time

print("First 10 predictions:")
print(y_reg_pred[:10])
print("\nPrediction time:", regression_prediction_time, "seconds")

First 10 predictions:
[0.6310295  0.7307619  0.81850995 0.70264775 0.77316318 0.80234996
 0.56685805 0.60256535 0.65569732 0.73926687]

Prediction time: 0.0002121000002262008 seconds


In [29]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


X_train_log = np.column_stack([np.ones(X_train_scaled.shape[0]),X_train_scaled])
X_test_log = np.column_stack([np.ones(X_test_scaled.shape[0]),X_test_scaled])

weights = np.zeros(X_train_log.shape[1])

learning_rate = 0.01
iterations = 5000

start_time = time.perf_counter()

for i in range(iterations):
    z = X_train_log @ weights
    probabilities = sigmoid(z)

    gradient = (
        X_train_log.T @ (probabilities - y_class_train)
    ) / len(y_class_train)

    weights = weights - learning_rate * gradient

classification_training_time = time.perf_counter() - start_time

print("Logistic Regression training time:",classification_training_time,"seconds")

Logistic Regression training time: 1.1273943999999574 seconds


In [30]:
start_time = time.perf_counter()
class_probabilities = sigmoid(X_test_log @ weights)

y_class_pred = (class_probabilities >= 0.5).astype(int)
classification_prediction_time = time.perf_counter() - start_time

errors = y_reg_test - y_reg_pred

mae = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors ** 2))
ss_res = np.sum(errors ** 2)
ss_total = np.sum((y_reg_test - np.mean(y_reg_train)) ** 2)
r2 = 1 - (ss_res / ss_total)
print("REGRESSION RESULTS")
print("--------------")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

print("\nClassification prediction time:", classification_prediction_time,"seconds")

REGRESSION RESULTS
--------------
MAE: 0.11211019963606006
RMSE: 0.15050415324738523
R2: 0.15236119481594335

Classification prediction time: 0.0004958999998052604 seconds


In [31]:
tp = np.sum((y_class_test == 1) & (y_class_pred == 1))
tn = np.sum((y_class_test == 0) & (y_class_pred == 0))
fp = np.sum((y_class_test == 0) & (y_class_pred == 1))
fn = np.sum((y_class_test == 1) & (y_class_pred == 0))

accuracy = (tp + tn) / len(y_class_test)
precision = tp / (tp + fp) if (tp + fp) != 0 else 0
recall = tp / (tp + fn) if (tp + fn) != 0 else 0
f1 = (2 * precision * recall / (precision + recall)
      if (precision + recall) != 0 else 0)

print("CLASSIFICATION RESULTS")
print("----------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

print("\nFinal Comparison")
print("----------------")
print("Linear Regression")
print("Training time:", regression_training_time)
print("Prediction time:", regression_prediction_time)
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

print("\nLogistic Regression")
print("Training time:", classification_training_time)
print("Prediction time:", classification_prediction_time)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

CLASSIFICATION RESULTS
----------------------
Accuracy: 0.7541666666666667
Precision: 0.7892156862745098
Recall: 0.9096045197740112
F1-score: 0.8451443569553806

Final Comparison
----------------
Linear Regression
Training time: 0.2233992000001308
Prediction time: 0.0002121000002262008
MAE: 0.11211019963606006
RMSE: 0.15050415324738523
R2: 0.15236119481594335

Logistic Regression
Training time: 1.1273943999999574
Prediction time: 0.0004958999998052604
Accuracy: 0.7541666666666667
Precision: 0.7892156862745098
Recall: 0.9096045197740112
F1-score: 0.8451443569553806


## Part C - Comparison and Optimization 


In [32]:
sklearn_reg_train = regression_training_time
sklearn_reg_pred = regression_prediction_time

sklearn_mae = mae
sklearn_rmse = rmse
sklearn_r2 = r2

sklearn_log_train = classification_training_time
sklearn_log_pred = classification_prediction_time

sklearn_accuracy = accuracy
sklearn_precision = precision
sklearn_recall = recall
sklearn_f1 = f1

print("Scikit-learn results saved successfully.")

Scikit-learn results saved successfully.


In [34]:
comparison = pd.DataFrame({
    "Implementation": ["Scikit-learn","Manual"],
    "Regression MAE": [sklearn_mae,mae],
    "Regression RMSE": [sklearn_rmse,rmse],
    "Regression R2": [sklearn_r2,r2],
    "Classification Accuracy": [sklearn_accuracy,accuracy],
    "Classification Precision": [sklearn_precision,precision],
    "Classification Recall": [sklearn_recall, recall],
    "Classification F1": [sklearn_f1,f1]})

print(comparison.round(10))

  Implementation  Regression MAE  Regression RMSE  Regression R2  \
0   Scikit-learn         0.11211         0.150504       0.152361   
1         Manual         0.11211         0.150504       0.152361   

   Classification Accuracy  Classification Precision  Classification Recall  \
0                 0.754167                  0.789216               0.909605   
1                 0.754167                  0.789216               0.909605   

   Classification F1  
0           0.845144  
1           0.845144  


In [36]:
time_comparison = pd.DataFrame({
    "Implementation": ["Scikit-learn","Manual"],
    "Linear Regression Training": [sklearn_reg_train,regression_training_time],
    "Linear Regression Prediction": [sklearn_reg_pred,regression_prediction_time],
    "Logistic Regression Training": [sklearn_log_train,classification_training_time],
    "Logistic Regression Prediction": [sklearn_log_pred,classification_prediction_time]})

print(time_comparison.round(10))

  Implementation  Linear Regression Training  Linear Regression Prediction  \
0   Scikit-learn                    0.223399                      0.000212   
1         Manual                    0.223399                      0.000212   

   Logistic Regression Training  Logistic Regression Prediction  
0                      1.127394                        0.000496  
1                      1.127394                        0.000496  


In [37]:
learning_rate = 0.05
iterations = 10000
lambda_reg = 0.01

weights_opt = np.zeros(X_train_log.shape[1])
start_time = time.perf_counter()

for i in range(iterations):
    z = X_train_log @ weights_opt
    probabilities = sigmoid(z)
    gradient = (X_train_log.T @ (probabilities - y_class_train)) / len(y_class_train)
    gradient[1:] = gradient[1:] + lambda_reg * weights_opt[1:]
    weights_opt = weights_opt - learning_rate * gradient

optimized_training_time = time.perf_counter() - start_time

print("Optimized Logistic Regression")
print("Training time:", optimized_training_time)

Optimized Logistic Regression
Training time: 3.957213899999715


In [38]:
start_time = time.perf_counter()
optimized_probabilities = sigmoid(X_test_log @ weights_opt)
optimized_pred = (optimized_probabilities >= 0.5).astype(int)
optimized_prediction_time = time.perf_counter() - start_time

tp = np.sum((y_class_test == 1) & (optimized_pred == 1))
tn = np.sum((y_class_test == 0) & (optimized_pred == 0))
fp = np.sum((y_class_test == 0) & (optimized_pred == 1))
fn = np.sum((y_class_test == 1) & (optimized_pred == 0))

optimized_accuracy = (tp + tn) / len(y_class_test)
optimized_precision = tp / (tp + fp) if (tp + fp) != 0 else 0
optimized_recall = tp / (tp + fn) if (tp + fn) != 0 else 0
optimized_f1 = (
    2 * optimized_precision * optimized_recall /(optimized_precision + optimized_recall)
    if (optimized_precision + optimized_recall) != 0 else 0)

print("Optimized Logistic Regression Results")
print("--------------------------------------")
print("Accuracy:", optimized_accuracy)
print("Precision:", optimized_precision)
print("Recall:", optimized_recall)
print("F1-score:", optimized_f1)
print("Training time:", optimized_training_time)
print("Prediction time:", optimized_prediction_time)

Optimized Logistic Regression Results
--------------------------------------
Accuracy: 0.7625
Precision: 0.7884615384615384
Recall: 0.9265536723163842
F1-score: 0.8519480519480519
Training time: 3.957213899999715
Prediction time: 0.0003394000000298547


In [39]:
optimization_comparison = pd.DataFrame({
    "Implementation": ["Scikit-learn","Manual","Manual Optimized"],
    "Accuracy": [sklearn_accuracy,accuracy,optimized_accuracy],
    "Precision": [sklearn_precision, precision, optimized_precision],
    "Recall": [sklearn_recall,recall,optimized_recall],
    "F1-score": [sklearn_f1,f1,optimized_f1],
    "Training Time": [sklearn_log_train,classification_training_time,optimized_training_time],
    "Prediction Time": [sklearn_log_pred,classification_prediction_time,optimized_prediction_time]
    })

print(optimization_comparison.round(6))

     Implementation  Accuracy  Precision    Recall  F1-score  Training Time  \
0      Scikit-learn  0.754167   0.789216  0.909605  0.845144       1.127394   
1            Manual  0.754167   0.789216  0.909605  0.845144       1.127394   
2  Manual Optimized  0.762500   0.788462  0.926554  0.851948       3.957214   

   Prediction Time  
0         0.000496  
1         0.000496  
2         0.000339  


The Scikit-learn and manual implementations were compared using the same train-test split and the same evaluation metrics. The manual implementation successfully reproduced Linear Regression and Logistic Regression using NumPy and Pandas.

The manual Logistic Regression was then optimized using vectorized NumPy operations, a tuned learning rate, increased iterations, and L2 regularization. The optimized model was compared with both the original manual implementation and the Scikit-learn implementation.

The execution time can differ because Scikit-learn uses highly optimized numerical implementations and efficient optimization solvers. The manual implementation is mainly intended to demonstrate the underlying mathematical workflow, so a runtime gap is expected.

The optimization improved the manual model by allowing better convergence and controlling the model parameters through regularization. The final metrics show how closely the manual implementation can reproduce the predictive performance of a standard machine-learning library.